# Test `newsindex` against `sample_articles.xlsx`

This notebook is a hand-checkable test of the `newsindex` library. `sample_articles.xlsx`
has 24 articles across 2 countries (Nigeria, Kenya), 2 newspapers each, and 2 months each
(3 articles per newspaper-month). Every article was written to hit a known, deliberate
combination of Economic / Policy / Uncertainty terms from the **real, production**
`eptu_country_keywords.json` dictionary bundled in this repo, and each row's
`expected_is_epu` label was verified against that dictionary (see
`scripts not included here, but every row's note_for_reviewer column names the exact terms it hits`).

Every newspaper-month has exactly 3 articles: 1 EPU article in the first month, 2 in the
second. That makes the whole pipeline -- classification, counting, baseline fitting, and
scoring -- computable by hand:

- `S_it` (EPU share) = 1/3 then 2/3 for every newspaper
- population std of `[1/3, 2/3]` = 1/6
- `N_it` = `S_it / sigma_i` = 2.0 then 4.0
- `mu` (baseline mean of `M_t`) = mean(2.0, 4.0) = **3.0**
- `EPU_t` = `100 * M_t / mu` = **66.67** in month 1, **133.33** in month 2, for *both* countries

This notebook asserts every one of those numbers against what the library actually
computes. If it runs with no `AssertionError`, the installed `newsindex` matches this
worked example exactly.

## Setup

If you installed the package with `pip install newsindex`, skip the `sys.path` line below
and just `import newsindex as ni`. Because `data/eptu_country_keywords.json` is not
bundled into the PyPI package (see the README's "Data" section), this notebook assumes
it is run from inside a clone of
[CAPE-ERC/newsindex-library](https://github.com/CAPE-ERC/newsindex-library), so it can
reach both the library source and `data/` by relative path.

In [1]:
import sys
from pathlib import Path

REPO_ROOT = Path.cwd().parent if Path.cwd().name == "examples" else Path.cwd()
sys.path.insert(0, str(REPO_ROOT / "src"))  # remove this line if newsindex is pip-installed

import pandas as pd
import newsindex as ni

print("newsindex", ni.__version__)

newsindex 0.1.0


## Load the sample articles

In [2]:
articles = pd.read_excel(REPO_ROOT / "examples" / "sample_articles.xlsx")
articles["year_month"] = articles["date"].astype(str).str.slice(0, 7)
articles.head()

,country,site,date,title,url,text,expected_is_epu,note_for_reviewer,year_month
0,Nigeria,The Punch,2024-01-01,Naira falls as inflation worsens amid fiscal p...,https://example.com/nigeria/the-punch/2024-01/01,The naira weakened further against the dollar ...,True,naira(E) inflation(E) fiscal policy(P) uncerta...,2024-01
1,Nigeria,The Punch,2024-01-02,Government announces new tax reform to boost e...,https://example.com/nigeria/the-punch/2024-01/02,The government on Monday unveiled a new tax re...,False,government(P) tax reform(P) economic growth(E)...,2024-01
2,Nigeria,The Punch,2024-01-03,Local football club wins championship match,https://example.com/nigeria/the-punch/2024-01/03,Thousands of fans packed the stadium on Sunday...,False,no E/P/U terms,2024-01
3,Nigeria,The Punch,2024-02-01,Central Bank of Nigeria raises interest rates ...,https://example.com/nigeria/the-punch/2024-02/01,The Central Bank of Nigeria raised its benchma...,True,central bank of nigeria(P) volatile(U) exchang...,2024-02
4,Nigeria,The Punch,2024-02-02,Inflation eases but fiscal policy keeps market...,https://example.com/nigeria/the-punch/2024-02/02,"Inflation slowed slightly last month, but trad...",True,inflation(E) fiscal policy(P) volatile(U),2024-02


## Step 1 -- classify: does `match.is_epu` agree with every hand-labeled `expected_is_epu`?

In [3]:
kw = ni.dictionary.load(REPO_ROOT / "data" / "eptu_country_keywords.json")
assert ni.dictionary.validate(kw) == [], "production dictionary should have no overlap/empty issues"

def classify(row):
    entry = kw[row["country"]]
    return ni.match.is_epu(row["title"], row["text"], entry.economic, entry.policy, entry.uncertainty)

articles["actual_is_epu"] = articles.apply(classify, axis=1)
mismatches = articles[articles["actual_is_epu"] != articles["expected_is_epu"]]
assert mismatches.empty, mismatches[["country", "site", "date", "title", "expected_is_epu", "actual_is_epu"]]
print(f"All {len(articles)} articles classified exactly as expected.")
articles[["country", "site", "date", "title", "expected_is_epu", "note_for_reviewer"]]

All 24 articles classified exactly as expected.


,country,site,date,title,expected_is_epu,note_for_reviewer
0,Nigeria,The Punch,2024-01-01,Naira falls as inflation worsens amid fiscal p...,True,naira(E) inflation(E) fiscal policy(P) uncerta...
1,Nigeria,The Punch,2024-01-02,Government announces new tax reform to boost e...,False,government(P) tax reform(P) economic growth(E)...
2,Nigeria,The Punch,2024-01-03,Local football club wins championship match,False,no E/P/U terms
3,Nigeria,The Punch,2024-02-01,Central Bank of Nigeria raises interest rates ...,True,central bank of nigeria(P) volatile(U) exchang...
4,Nigeria,The Punch,2024-02-02,Inflation eases but fiscal policy keeps market...,True,inflation(E) fiscal policy(P) volatile(U)
5,Nigeria,The Punch,2024-02-03,President signs new law amid political debate,False,law(P) only -- no E or U term
6,Nigeria,Vanguard,2024-01-01,Fiscal deficit widens as government struggles ...,True,fiscal deficit(E) government(P) crisis(U)
7,Nigeria,Vanguard,2024-01-02,Ministry of Finance unveils new regulation,False,ministry of finance(P) regulation(P) -- policy...
8,Nigeria,Vanguard,2024-01-03,Market risk rises amid unstable investment cli...,False,risk(U) unstable(U) investment(E) -- no policy...
9,Nigeria,Vanguard,2024-02-01,GDP growth slows as monetary policy tightens a...,True,GDP(E) monetary policy(P) instability(U)


## Step 2 -- deduplicate (a no-op here: every URL is unique) and count per newspaper-month

In [4]:
deduped = ni.dedup.dedupe(articles)
assert len(deduped) == len(articles)

counts = ni.index.count(deduped, kw)
counts

,country,site,year_month,epu_articles,all_articles
0,Kenya,Daily Nation,2024-01,1,3
1,Kenya,Daily Nation,2024-02,2,3
2,Kenya,The Star,2024-01,1,3
3,Kenya,The Star,2024-02,2,3
4,Nigeria,The Punch,2024-01,1,3
5,Nigeria,The Punch,2024-02,2,3
6,Nigeria,Vanguard,2024-01,1,3
7,Nigeria,Vanguard,2024-02,2,3


Expect `epu_articles = [1, 2]` and `all_articles = 3` for every one of the 4
newspapers' 2 months.

In [5]:
assert set(counts["all_articles"]) == {3}
assert sorted(counts["epu_articles"]) == [1, 1, 1, 1, 2, 2, 2, 2]

## Step 3 -- fit the frozen baseline over both months

In [6]:
baseline = ni.index.fit_baseline(counts, start="2024-01", end="2024-02")

for country in ("Nigeria", "Kenya"):
    assert baseline.mu[country] == 3.0, baseline.mu
    for site, sigma in baseline.sigma[country].items():
        assert abs(sigma - 1 / 6) < 1e-9, (country, site, sigma)

print("mu:", baseline.mu)
print("sigma:", baseline.sigma)

mu: {'Kenya': 3.0, 'Nigeria': 3.0}
sigma: {'Kenya': {'Daily Nation': 0.16666666666666666, 'The Star': 0.16666666666666666}, 'Nigeria': {'The Punch': 0.16666666666666666, 'Vanguard': 0.16666666666666666}}


## Step 4 -- score both months

In [7]:
scored = ni.index.score(counts, baseline).set_index(["country", "year_month"])

for country in ("Nigeria", "Kenya"):
    assert abs(scored.loc[(country, "2024-01"), "EPU"] - 200 / 3) < 1e-6
    assert abs(scored.loc[(country, "2024-02"), "EPU"] - 400 / 3) < 1e-6

print("All EPU values match the hand-computed 66.67 / 133.33.")
scored.reset_index()

All EPU values match the hand-computed 66.67 / 133.33.


,country,year_month,EPU,M_t,mu,n_sites
0,Kenya,2024-01,66.666667,2.0,3.0,2
1,Kenya,2024-02,133.333333,4.0,3.0,2
2,Nigeria,2024-01,66.666667,2.0,3.0,2
3,Nigeria,2024-02,133.333333,4.0,3.0,2


## Step 5 -- (optional) baseline persistence and a trivial two-country aggregate

This exercises `Baseline.save`/`Baseline.load` and `aggregate.aggregate` on the same
tiny example, with Nigeria and Kenya given equal, constant GDP weight so the regional
z-score is just the simple average of the two (identical) country z-score series --
another number you can check by hand.

In [8]:
import tempfile

with tempfile.TemporaryDirectory() as tmp:
    baseline_path = Path(tmp) / "baseline.json"
    baseline.save(baseline_path)
    reloaded = ni.index.Baseline.load(baseline_path)
    assert reloaded.mu == baseline.mu and reloaded.sigma == baseline.sigma

gdp = pd.DataFrame({
    "country": ["Nigeria", "Nigeria", "Kenya", "Kenya"],
    "year_month": ["2024-01", "2024-02", "2024-01", "2024-02"],
    "gdp": [50.0, 50.0, 50.0, 50.0],
})

regions = ni.aggregate.aggregate(
    scored.reset_index(),
    gdp,
    groups={"TestRegion": ["Nigeria", "Kenya"]},
    baseline=reloaded,
    include_africa=False,
)
regions

,group,year_month,z,index,n_countries,gdp_covered
0,TestRegion,2024-01,-1.0,90.0,2,100.0
1,TestRegion,2024-02,1.0,110.0,2,100.0


## Done

Every assertion above passed, which means the installed `newsindex` reproduces this
worked example exactly: classification, newspaper-month counting, baseline fitting
(`sigma`, `mu`), scoring (`EPU_t`), baseline persistence, and GDP-weighted aggregation.